# Multi-Agent Supervisor Architecture: Practice Exercise

Build a content creation agency using LangGraph's supervisor pattern. You'll create specialized agents that a supervisor orchestrates to research topics, write content, and generate promotional materials.

**What you'll implement:**
- The supervisor agent that routes tasks to specialized workers
- The LangGraph workflow connecting all agents

**Estimated time:** 15 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [ ]:
# Setup - run this cell first

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

from typing import Annotated, Literal, Sequence
from typing_extensions import TypedDict
from pydantic import BaseModel
import operator
import functools

from dotenv import load_dotenv
load_dotenv()

print("Setup complete!")

## Model Configuration

Initialize the language model that all agents will use.

In [ ]:
# Define the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

print("Model initialized!")

## Tools (Provided)

These tools are ready to use with your agents.

In [ ]:
# Tool 1: Web Search (for the Research Agent)
tavily_search = TavilySearchResults(max_results=5)

# Tool 2: Content Database Lookup (for the Writer Agent)
@tool
def get_content_guidelines(content_type: str) -> str:
    """
    Retrieves content guidelines and best practices for a specific content type.
    
    Args:
        content_type: The type of content (blog, social, email, whitepaper)
    
    Returns:
        Guidelines and best practices for the content type
    """
    guidelines = {
        "blog": "Blog posts should be 800-1500 words, include headers every 200-300 words, "
                "use conversational tone, include a compelling intro hook, and end with a clear CTA.",
        "social": "Social posts should be concise (under 280 chars for Twitter, under 2200 for LinkedIn), "
                  "include relevant hashtags, use engaging hooks, and include a call-to-action.",
        "email": "Emails should have compelling subject lines under 50 chars, personalized greeting, "
                 "scannable format with bullet points, and a single clear CTA.",
        "whitepaper": "Whitepapers should be 2500-5000 words, include executive summary, "
                      "data-driven insights, professional tone, and actionable conclusions."
    }
    return guidelines.get(content_type.lower(), "General guidelines: Be clear, concise, and audience-focused.")

# Tool 3: SEO Keyword Generator (for the SEO Agent)
@tool
def generate_seo_keywords(topic: str, target_audience: str) -> str:
    """
    Generates SEO keywords and meta description suggestions for a topic.
    
    Args:
        topic: The main topic or subject
        target_audience: The intended audience for the content
    
    Returns:
        SEO keywords and meta description suggestions
    """
    return f"""SEO Analysis for '{topic}' targeting '{target_audience}':
    
Primary Keywords: {topic}, {topic} guide, {topic} tips, {topic} for {target_audience}
Secondary Keywords: best {topic}, {topic} strategies, how to {topic}, {target_audience} {topic}
Long-tail Keywords: complete guide to {topic}, {topic} best practices for {target_audience}

Meta Description Template: Discover expert {topic} strategies designed for {target_audience}. 
Learn actionable tips and best practices to achieve your goals.

Recommended: Use primary keyword in title, first paragraph, and at least 2 headers."""

print("Tools loaded successfully!")
print("Available tools: tavily_search, get_content_guidelines, generate_seo_keywords")

## Agents (Provided)

Three specialized agents are pre-built for you. Your task is to create the supervisor that orchestrates them.

In [ ]:
# Define system prompts for each agent
AGENT_PROMPTS = {
    "ResearchAgent": (
        "You are a research specialist. Use web search tools to gather information on topics. "
        "Return ONLY raw research findings: facts, statistics, quotes, and sources. "
        "Do NOT write outlines, drafts, or any formatted content. "
        "Do NOT create blog posts or articles. Just provide the research data for others to use."
    ),
    "WriterAgent": (
        "You are a professional content writer. Your job is to take research findings from the "
        "conversation and create the actual written content (blog outline, draft, article, etc.). "
        "Use the content guidelines tool to check best practices for the content type. "
        "Always produce the specific deliverable requested (outline, draft, post, etc.)."
    ),
    "SEOAgent": (
        "You are an SEO specialist. Analyze the content topic and provide SEO optimization "
        "recommendations. Use the keyword generator tool, then provide actionable SEO advice "
        "based on the results."
    )
}

# Research Agent - gathers information on topics
research_agent = create_agent(
    model=llm,
    tools=[tavily_search]
)

# Writer Agent - creates content based on research
writer_agent = create_agent(
    model=llm,
    tools=[get_content_guidelines]
)

# SEO Agent - optimizes content for search
seo_agent = create_agent(
    model=llm,
    tools=[generate_seo_keywords]
)

print("Agents created!")
print("Available agents: research_agent, writer_agent, seo_agent")

## Context

You are building a content creation agency with a supervisor architecture:

**Supervisor**: Routes tasks to the appropriate specialist agent based on the current needs:
- Routes research requests to **ResearchAgent**
- Routes content writing tasks to **WriterAgent**
- Routes SEO optimization to **SEOAgent**
- Decides when the task is complete and responds with **FINISH**

**Workflow**: 
1. User makes a request (e.g., "Write a blog post about AI in healthcare")
2. Supervisor analyzes and routes to ResearchAgent first
3. ResearchAgent gathers information, reports back to Supervisor
4. Supervisor routes to WriterAgent with the research
5. WriterAgent creates content, reports back to Supervisor
6. Supervisor may route to SEOAgent for optimization
7. Supervisor decides task is complete, responds with FINISH

**Your tasks:**
1. Define the supervisor agent with proper routing logic
2. Build the LangGraph workflow connecting all components

## State Definition (Provided)

The state structure that flows through the graph.

In [ ]:
# State definition for the multi-agent system
class AgentState(TypedDict):
    messages: Annotated[Sequence[HumanMessage], operator.add]
    next: str

print("State defined!")

## Agent Node Helper (Provided)

A helper function that wraps agent execution for use as graph nodes.

In [ ]:
from langchain_core.messages import SystemMessage

def agent_node(state: AgentState, agent, name: str) -> dict:
    """
    Executes an agent and formats its response for the graph.
    Injects a system prompt to give the agent role-specific context.
    
    Args:
        state: Current graph state with messages
        agent: The agent to execute
        name: Name to assign to the agent's response
    
    Returns:
        Dictionary with the agent's response as a named AIMessage
    """
    # Inject system prompt for this agent's role
    system_prompt = AGENT_PROMPTS.get(name, "You are a helpful assistant.")
    messages_with_context = [SystemMessage(content=system_prompt)] + list(state["messages"])
    
    result = agent.invoke({"messages": messages_with_context})
    return {
        "messages": [AIMessage(content=result["messages"][-1].content, name=name)]
    }

# Create node functions for each agent
research_node = functools.partial(agent_node, agent=research_agent, name="ResearchAgent")
writer_node = functools.partial(agent_node, agent=writer_agent, name="WriterAgent")
seo_node = functools.partial(agent_node, agent=seo_agent, name="SEOAgent")

print("Agent nodes created!")
print("Available nodes: research_node, writer_node, seo_node")

## Part 1: Define the Supervisor Agent

Create the supervisor that decides which agent should handle the next task. The supervisor needs:

1. **Team member definitions** - A dictionary mapping agent names to their descriptions
2. **Routing options** - The possible next steps (agent names + "FINISH")
3. **Output schema** - A Pydantic model for structured routing decisions
4. **Prompt template** - Instructions for the supervisor's decision-making
5. **Supervisor function** - A function that takes state and returns the routing decision

In [ ]:
# TODO 1: Define the team members dictionary
# Map agent names to descriptions of their capabilities
# Keys should be: "ResearchAgent", "WriterAgent", "SEOAgent"

team_members = {
    # TODO: Add entries for each agent
    # "AgentName": "Description of what this agent does"
}

# TODO 2: Define the routing options
# Should include "FINISH" plus all team member names

routing_options = []  # TODO: Create list with "FINISH" and all team member keys

print(f"Team members: {list(team_members.keys())}")
print(f"Routing options: {routing_options}")

In [ ]:
# TODO 3: Define the output schema for routing decisions
# This Pydantic model ensures the supervisor returns a valid routing choice

class SupervisorDecision(BaseModel):
    """
    The supervisor's routing decision.
    
    Attributes:
        next: The next agent to call, or "FINISH" if the task is complete.
              Must be one of: "FINISH", "ResearchAgent", "WriterAgent", "SEOAgent"
    """
    # TODO: Add a 'next' field with a Literal type containing valid routing options
    # Hint: Use Literal["FINISH", "ResearchAgent", "WriterAgent", "SEOAgent"]
    pass

In [ ]:
# TODO 4: Create the supervisor prompt template
# The prompt should instruct the supervisor to:
# - Analyze the user's request and conversation history
# - Choose which agent should handle the next task
# - Return "FINISH" when all objectives are met

supervisor_system_prompt = """
You are a supervisor managing a content creation team with these specialists:
{members_description}

YOUR JOB: Route tasks to the right specialist and decide when the user's request is complete.

DECISION PROCESS:
1. Look at the user's ORIGINAL request - what did they ask for?
2. Review the conversation - what has been delivered so far?
3. Ask yourself: "Has the user's request been fully satisfied?"
   - If YES -> choose FINISH
   - If NO -> route to the specialist who can complete the remaining work

GUIDELINES:
- ResearchAgent gathers information but does NOT write final content
- WriterAgent creates the actual written deliverables
- SEOAgent provides keyword and optimization recommendations
- Each agent should only be called once per task
- Choose FINISH when the deliverable the user asked for has been produced
"""

# Format the members description
members_description = "\n".join([f"- {name}: {desc}" for name, desc in team_members.items()])

# TODO: Create the ChatPromptTemplate with:
# - A system message containing the supervisor_system_prompt (formatted with members_description)
# - A MessagesPlaceholder for "messages" (the conversation history)
# - A final system message asking "Who should act next? Choose from: {options}"

supervisor_prompt = None  # TODO: Create ChatPromptTemplate.from_messages([...])

print("Supervisor prompt created!")

In [ ]:
# TODO 5: Create the supervisor agent function

def supervisor_agent(state: AgentState) -> dict:
    """
    The supervisor agent that routes tasks to specialized agents.
    
    Args:
        state: Current graph state containing messages
    
    Returns:
        Dictionary with 'next' key indicating which agent should act
    """
    # TODO: Implement the supervisor logic:
    # 1. Create a chain: supervisor_prompt | llm.with_structured_output(SupervisorDecision)
    # 2. Invoke the chain with the state
    # 3. Return the result (it will be a dict with 'next' key)
    pass

print("Supervisor agent function defined!")

## Part 2: Build the LangGraph Workflow

Create the graph that connects the supervisor with the specialist agents. The workflow should:

1. Start at the Supervisor
2. Supervisor routes to an agent (or FINISH)
3. Each agent reports back to the Supervisor
4. Supervisor decides next step until FINISH

In [ ]:
# TODO: Build the complete LangGraph workflow

# Step 1: Initialize the StateGraph with AgentState
workflow = None  # TODO: StateGraph(AgentState)

# Step 2: Add nodes for each agent and the supervisor
# TODO: Add nodes using workflow.add_node(name, function)
# Nodes needed: "ResearchAgent", "WriterAgent", "SEOAgent", "Supervisor"


# Step 3: Add edges from each agent back to the Supervisor
# TODO: Add edges using workflow.add_edge(from_node, to_node)
# Each specialist agent should report back to "Supervisor"


# Step 4: Add conditional edges from Supervisor to agents or END
# TODO: Create a routing map and add conditional edges
# The routing map should map each agent name to itself, and "FINISH" to END
# Use workflow.add_conditional_edges("Supervisor", lambda x: x["next"], routing_map)


# Step 5: Set the entry point to Supervisor
# TODO: Add edge from START to "Supervisor"


# Step 6: Compile the graph with memory checkpointing
memory = MemorySaver()
graph = None  # TODO: workflow.compile(checkpointer=memory)

print("Workflow compiled!")

## Visualize the Graph

Display the workflow structure to verify your implementation.

In [ ]:
from IPython.display import display, Image

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not visualize graph: {e}")

## Test Your Implementation

Run these test cases to verify your multi-agent system works correctly.

In [ ]:
# Helper function to process and display events
def process_event(event: dict) -> None:
    """Display information about a graph event."""
    for node_name, node_output in event.items():
        if node_name == "__end__":
            continue
            
        # Handle supervisor routing decisions
        if "next" in node_output:
            print(f">>> Supervisor routes to: {node_output['next']}")
            print("-" * 40)
        
        # Handle agent messages
        if "messages" in node_output:
            for message in node_output["messages"]:
                agent_name = getattr(message, "name", node_name)
                content = message.content
                print(f"\n{'='*60}")
                print(f"Agent: {agent_name}")
                print(f"{'='*60}")
                # Show full content or truncate if very long
                if len(content) > 1500:
                    print(f"{content[:1500]}...")
                    print(f"\n[Output truncated - {len(content)} total characters]")
                else:
                    print(content)
                print()

print("Test helper ready!")

In [ ]:
# Test 1: Simple content request
print("TEST 1: Blog Post Request")
print("=" * 60)

config = {"configurable": {"thread_id": "test-1"}}

events = graph.stream(
    {"messages": [HumanMessage(content="Research the topic of 'remote work productivity tips', then write a short blog post about it.")]},
    config=config
)

for event in events:
    process_event(event)

print("Test 1 complete!")

In [ ]:
# Test 2: SEO-focused request
print("TEST 2: SEO Keywords Request")
print("=" * 60)

config = {"configurable": {"thread_id": "test-2"}}

events = graph.stream(
    {"messages": [HumanMessage(content="Generate SEO keywords for an article about 'sustainable fashion' targeting eco-conscious millennials.")]},
    config=config
)

for event in events:
    process_event(event)

print("Test 2 complete!")